In [12]:
import torch
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [13]:
x=torch.arange(5,dtype=torch.float32)
print(x)
x.requires_grad_(True)
x.grad
y=2*torch.dot(x,x)
y
y.backward()
x.grad
x.grad==4*x

tensor([0., 1., 2., 3., 4.])


tensor([0., 1., 2., 3., 4.], requires_grad=True)

tensor(60., grad_fn=<MulBackward0>)

tensor([ 0.,  4.,  8., 12., 16.])

tensor([True, True, True, True, True])

In [14]:
x.grad.zero_()
y=x.sum()
y.backward()
x.grad

tensor([0., 0., 0., 0., 0.])

tensor([1., 1., 1., 1., 1.])

In [15]:
#深度学习一般只针对一个标量函数（即cost func）做反向传播，若y为向量一般先sum再backward，因为y，x都是向量进行反向传播计算量暴增，且计算这个微分矩阵与目标无关
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward()
x.grad

tensor([0., 0., 0., 0., 0.])

tensor([0., 2., 4., 6., 8.])

In [ ]:
#中途希望某个参数不参与计算图的方法
x.grad.zero_()
y=x*x
u=y.detach()
z=u*y
z.sum().backward()
x.grad

tensor([0., 0., 0., 0., 0.])

tensor([ 0.,  1.,  4.,  9., 16.])

tensor([  0.,   2.,  16.,  54., 128.])

In [20]:
#由于pytorch的自动微分计算图是动态生成的，因此可以通过控制流来改变计算图的结构
def f(a):
    b=a*2
    while b.norm()<1000:
        b=b*2
    if b.sum()>0:
        c=b
    else:
        c=100*b
    return c

a=torch.randn(size=(),dtype=torch.float32,requires_grad=True)
c=f(a)
c.sum().backward()
a.grad==c/a


tensor(True)

In [29]:
print(help(torch.arange))
print(dir(torch.tensor))

Help on built-in function arange in module torch:

arange(...)
    arange(start=0, end, step=1, *, out=None, dtype=None, layout=torch.strided, device=None, requires_grad=False) -> Tensor
    
    Returns a 1-D tensor of size :math:`\left\lceil \frac{\text{end} - \text{start}}{\text{step}} \right\rceil`
    with values from the interval ``[start, end)`` taken with common difference
    :attr:`step` beginning from `start`.
    
    Note: When using floating-point dtypes (especially reduced precision types like ``bfloat16``),
    the results may be affected by floating-point rounding behavior. Some values in the sequence
    might not be exactly representable in certain floating-point formats, which can lead to
    repeated values or unexpected rounding. For precise sequences, it is recommended to use
    integer dtypes instead of floating-point dtypes.
    
    Note that non-integer :attr:`step` is subject to floating point rounding errors when
    comparing against :attr:`end`; to avoid